# Sequence lab 2 · Explain the Transformer forecaster: which channel, which moment, and is attention the answer?

**Author:** Xiangqi Kong

**Follows:** the earlier lab *Transformer for Time Series* (same building blocks, now in PyTorch) and *Sequence lab 1* · **Decks:** Post-hoc Model-Agnostic (*PCI*), Post-hoc Model-Specific (*Integrated Gradients*, *Attention*) · **Runtime:** about 2 minutes on a T4, about 4 minutes on CPU · **GPU:** helps

In the earlier lab you built an encoder-decoder Transformer that reads **100 time steps of 10 channels** and forecasts the future. In that data every channel was generated on its own, so there was nothing to discover about how channels depend on each other.

Here the data gets a **known answer**, built in on purpose:

- **Channel 0, the leader,** carries sudden bumps that nothing can predict.
- **Channel 1, the follower,** is the leader **20 steps later**. So the next 20 steps of the follower are already visible, in the leader.
- **Channels 2–8** are independent waves, as in the earlier lab.
- **Channel 9** is pure noise: a control that no forecast should depend on.

A model that forecasts well **must** use this structure. So we know what a correct explanation looks like, and we can mark each method right or wrong:

1. **PCI** (model-agnostic): which input channel does each forecast need?
2. **Integrated Gradients** (model-specific): the same question for single forecasts, and then *which moment* of the leader the follower's forecast reads.
3. **Attention**: does the cross-attention map show the 20-step delay? Is the map an explanation?

Before you run anything, write down your guess: *will the attention map show the 20-step delay?*

> **Easy version.** Almost all the code is written. Each **✏️ TODO** has exactly **one blank, written `BLANK`**, and the comment above it
> says what goes there. Replace `BLANK`, run the cell, then run the **✅ check** cell after it. If you see
> `NameError: name 'BLANK' is not defined`, a blank is still empty. The `_walkthrough` notebook in this folder explains every line; the `_solution` notebook has all the answers.

> **In the lab session: about 30 minutes.** Do the sections marked 🟢 core in order; they are what we discuss at the end.
> Sections marked ⚪ if time can be skipped: no 🟢 core section needs them, so come back to them at home.
>
> 🟢 core: all sections: PCI (TODO 1), IG (TODO 2), attention and the replace-the-attention test (TODO 3).  
> ⚪ if time: the Try it ideas at the end.

## 0. Setup

> **What this cell does.** It loads the libraries, fixes the random seed and checks for a GPU. Just run it.

In [ ]:
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from scipy.stats import spearmanr

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update(
    {"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110}
)
BLUE, RED, GREY = "#2563EB", "#DC2626", "#9CA3AF"
print("Device:", DEVICE, "| torch", torch.__version__)

## 1. The data: the earlier lab's waves, plus a known answer · 🟢 core

The generator follows the earlier notebook: each channel is a mix of 2–3 sine waves, a small trend and noise. Four things are changed, and each is marked in the code:

| Change | Why |
|---|---|
| The leader gets random **bumps** | a part of the signal that cannot be forecast from its own past |
| The follower **copies the leader 20 steps later** | the only way to forecast its bumps is to read the leader: the known answer |
| Channel 9 is **pure noise** | a control: its importance must be zero (the NOISE channel of the PCI lab) |
| Slower waves, and a forecast of **20 steps** (was 50) | so that a small model trains well in about a minute |

> **What this cell does.** It generates the data: 10 channels over time. Channel 0 (the leader) has random bumps; channel 1 (the follower) repeats the leader 20 steps later; channels 2 to 8 are separate waves; channel 9 is noise. Just run it.

In [ ]:
N_SAMPLES, IN_STEPS, OUT_STEPS, D_IN = 20000, 100, 20, 10
LEADER, FOLLOWER, NOISE, LAG = 0, 1, 9, 20
CHANNELS = (
    ["0 leader", "1 follower"]
    + [f"{d} wave" for d in range(2, 9)]
    + ["9 noise"]
)


def make_data(n=N_SAMPLES, seed=SEED):
    rng = np.random.default_rng(seed)
    total = IN_STEPS + OUT_STEPS
    # LAG extra steps at the start, for the follower to copy
    t = np.arange(-LAG, total)
    # mix of 2-3 sine waves + trend + offset per channel, as in the earlier lab
    # (slower waves)
    k = rng.integers(2, 4, (n, 1, D_IN, 1))
    amp = rng.uniform(0.3, 1.2, (n, 1, D_IN, 3)) * (np.arange(3) < k)
    freq = rng.uniform(0.004, 0.02, (n, 1, D_IN, 3))
    phase = rng.uniform(0, 2 * np.pi, (n, 1, D_IN, 3))
    s = (amp * np.sin(2 * np.pi * freq * t[None, :, None, None] + phase)).sum(
        -1
    )
    s += rng.uniform(-0.01, 0.01, (n, 1, D_IN)) * t[None, :, None] + rng.normal(
        0, 0.1, (n, 1, D_IN)
    )
    # NEW: 6 bumps at random moments on the leader
    centre = rng.uniform(-LAG, total, (n, 6, 1))
    height = rng.uniform(1.0, 2.0, (n, 6, 1)) * rng.choice([-1, 1], (n, 6, 1))
    s[:, :, LEADER] += (
        height * np.exp(-0.5 * ((t[None, None, :] - centre) / 2.5) ** 2)
    ).sum(1)
    s += rng.normal(0, 0.05, s.shape)
    data = s[:, LAG:].copy()
    # NEW: the leader, LAG steps later
    data[:, :, FOLLOWER] = s[:, :total, LEADER] + rng.normal(
        0, 0.05, (n, total)
    )
    # NEW: pure noise control
    data[:, :, NOISE] = rng.normal(0, 1.0, (n, total))
    return torch.tensor(data, dtype=torch.float32)


data = make_data().to(DEVICE)
# (N, 100, 10) history, (N, 20, 10) future
X, Y = data[:, :IN_STEPS], data[:, IN_STEPS:]
n_train, n_val = int(0.8 * N_SAMPLES), int(0.1 * N_SAMPLES)
X_train, Y_train = X[:n_train], Y[:n_train]
X_test, Y_test = X[n_train + n_val :], Y[n_train + n_val :]
print(
    "X_train",
    tuple(X_train.shape),
    "| X_test",
    tuple(X_test.shape),
    "| Y_test",
    tuple(Y_test.shape),
)

fig, axes = plt.subplots(3, 1, figsize=(11, 5.6), sharex=True)
t_in, t_out = np.arange(IN_STEPS), np.arange(IN_STEPS, IN_STEPS + OUT_STEPS)
for ax, ch in zip(axes, [LEADER, FOLLOWER, NOISE]):
    ax.plot(t_in, X_test[0, :, ch].cpu(), color=GREY)
    ax.plot(t_out, Y_test[0, :, ch].cpu(), color=BLUE)
    ax.axvline(IN_STEPS - 0.5, color="k", lw=0.6, ls=":")
    ax.set_ylabel(CHANNELS[ch])
axes[0].axvspan(IN_STEPS - LAG, IN_STEPS - 1, color=RED, alpha=0.12)
axes[1].axvspan(IN_STEPS, IN_STEPS + OUT_STEPS - 1, color=RED, alpha=0.12)
axes[0].set_title(
    "One test series. Grey = history the model sees, blue = future to forecast. The two red spans hold the same values.",
    fontsize=10,
)
axes[2].set_xlabel("time step")
plt.tight_layout()
plt.show()

> **What to notice.** In the plot, every bump of the leader appears again in the follower 20 steps later. This is the built-in answer that a good explanation must find.

## 2. The model: the earlier lab's Transformer, in PyTorch · 🟢 core

The blocks are the ones you wrote in the earlier lab: token embedding, sinusoidal positions, multi-head attention, feed-forward, pre-norm encoder and decoder blocks. Three small differences:

- **The attention layer remembers its weights** (`last_attn`), and lets us **replace** them (`override`). That is all the attention experiments need.
- **The decoder input is all zeros**, so each forecast step knows only its position. The earlier lab fed in the shifted target (teacher forcing) and then generated one step at a time. On this data the errors of that loop build up, and one forward pass is easier to explain.
- **Size:** `d_model = 32`, 4 heads, one encoder block and one decoder block.

> **What this cell does.** **You can skip reading this cell; just run it.** It builds the Transformer from your earlier lab in PyTorch and trains it for about a minute.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pos = torch.arange(max_len)[:, None]
        i = torch.arange(d_model)[None, :]
        ang = pos / torch.pow(10000.0, (2 * (i // 2)) / d_model)
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(ang[:, 0::2])
        pe[:, 1::2] = torch.cos(ang[:, 1::2])
        self.register_buffer("pe", pe[None])

    def forward(self, x):
        return x + self.pe[:, : x.shape[1]]


class MultiHeadAttention(nn.Module):
    # Self-attention if kv is None, cross-attention otherwise (queries from x,
    # keys and values from kv).
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.h, self.d = num_heads, d_model // num_heads
        self.q, self.k, self.v, self.proj = (
            nn.Linear(d_model, d_model) for _ in range(4)
        )
        # the weights of the last forward pass, shape (batch, heads, queries,
        # keys)
        self.last_attn = None
        # a function that replaces the weights, for the tests in section 7
        self.override = None

    def forward(self, x, kv=None, mask=None):
        src = x if kv is None else kv
        B, Tq, Tk = x.shape[0], x.shape[1], src.shape[1]
        # (B, heads, T, d)
        split = lambda z, T: z.view(B, T, self.h, self.d).transpose(1, 2)
        q, k, v = (
            split(self.q(x), Tq),
            split(self.k(src), Tk),
            split(self.v(src), Tk),
        )
        # 1. score
        att = q @ k.transpose(-1, -2) * self.d**-0.5
        if mask is not None:
            att = att.masked_fill(mask, -1e9)
        # 2. normalise
        att = att.softmax(-1)
        if self.override is not None:
            att = self.override(att)
        self.last_attn = att.detach()
        # 3. mix
        return self.proj((att @ v).transpose(1, 2).reshape(B, Tq, -1))


class FeedForward(nn.Sequential):
    def __init__(self, d, ratio, p):
        super().__init__(
            nn.Linear(d, d * ratio),
            nn.GELU(),
            nn.Dropout(p),
            nn.Linear(d * ratio, d),
            nn.Dropout(p),
        )


class EncoderBlock(nn.Module):
    def __init__(self, d, h, ratio, p):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = MultiHeadAttention(d, h)
        self.ffn = FeedForward(d, ratio, p)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.ffn(self.norm2(x))


class DecoderBlock(nn.Module):
    def __init__(self, d, h, ratio, p):
        super().__init__()
        self.norm1, self.norm2, self.norm3 = (
            nn.LayerNorm(d),
            nn.LayerNorm(d),
            nn.LayerNorm(d),
        )
        self.self_attn, self.cross_attn = MultiHeadAttention(
            d, h
        ), MultiHeadAttention(d, h)
        self.ffn = FeedForward(d, ratio, p)

    def forward(self, x, memory, tgt_mask):
        # masked self-attention (causal)
        x = x + self.self_attn(self.norm1(x), mask=tgt_mask)
        # cross-attention: forecast steps look at the history
        x = x + self.cross_attn(self.norm2(x), kv=memory)
        return x + self.ffn(self.norm3(x))


class TimeSeriesTransformer(nn.Module):
    def __init__(
        self,
        d_model=32,
        num_heads=4,
        depth_enc=1,
        depth_dec=1,
        mlp_ratio=2,
        dropout=0.1,
        d_in=D_IN,
    ):
        super().__init__()
        self.enc_tok, self.dec_tok = nn.Linear(d_in, d_model), nn.Linear(
            d_in, d_model
        )
        self.pos = PositionalEncoding(d_model)
        self.drop = nn.Dropout(dropout)
        self.encoders = nn.ModuleList(
            EncoderBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(depth_enc)
        )
        self.decoders = nn.ModuleList(
            DecoderBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(depth_dec)
        )
        self.head = nn.Linear(d_model, d_in)

    def forward(self, enc, dec_in):
        memory = self.drop(self.pos(self.enc_tok(enc)))
        for blk in self.encoders:
            memory = blk(memory)
        T = dec_in.shape[1]
        mask = torch.triu(
            torch.ones(T, T, dtype=torch.bool, device=dec_in.device), 1
        )
        x = self.drop(self.pos(self.dec_tok(dec_in)))
        for blk in self.decoders:
            x = blk(x, memory, mask)
        return self.head(x)


def forecast(enc):
    # History (B, 100, 10) -> forecast (B, 20, 10), in one forward pass.
    # Differentiable, so IG can use it.
    return model(
        enc,
        torch.zeros(enc.shape[0], OUT_STEPS, enc.shape[2], device=enc.device),
    )


EPOCHS, BATCH, LR = 12, 128, 2e-3
model = TimeSeriesTransformer().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), LR)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, LR, total_steps=EPOCHS * ((n_train + BATCH - 1) // BATCH)
)
t0 = time.time()
for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for b in torch.randperm(n_train, device=DEVICE).split(BATCH):
        loss = nn.functional.mse_loss(forecast(X_train[b]), Y_train[b])
        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()
        total += loss.item() * len(b)
    print(
        f"epoch {epoch + 1:2d}/{EPOCHS}   training MSE {total / n_train:.3f}   ({time.time() - t0:.0f} s)"
    )
model.eval()
# from now on we only need gradients with respect to the input
for p in model.parameters():
    p.requires_grad_(False)
# the layer whose weights we will look at
CROSS = model.decoders[0].cross_attn

### How good is the forecast?

*Predict the mean* is the error of a model that knows nothing. Three things to expect: the **follower** is forecast almost perfectly (its future is on show in the leader), the **noise** channel cannot be forecast at all, and the **leader** is hard because its future bumps are unknown.

In [ ]:
@torch.no_grad()
def channel_mse(Xin, Ytrue=Y_test):
    # Test MSE of every output channel, shape (10,).
    return ((forecast(Xin) - Ytrue) ** 2).mean((0, 1)).cpu().numpy()


BASE = channel_mse(X_test)
display(
    pd.DataFrame(
        {
            "Predict the mean": ((Y_test - Y_train.mean((0, 1))) ** 2)
            .mean((0, 1))
            .cpu()
            .numpy(),
            "Transformer": BASE,
        },
        index=CHANNELS,
    )
    .round(3)
    .T
)

with torch.no_grad():
    P_test = forecast(X_test[:8]).cpu()
fig, axes = plt.subplots(1, 4, figsize=(15, 3), sharey=True)
for ax, ch in zip(axes, [LEADER, FOLLOWER, 4, NOISE]):
    ax.plot(t_in[60:], X_test[0, 60:, ch].cpu(), color=GREY)
    ax.plot(t_out, Y_test[0, :, ch].cpu(), color="k", lw=1, label="true future")
    ax.plot(t_out, P_test[0, :, ch], color=RED, label="forecast")
    ax.set_title(CHANNELS[ch], fontsize=10)
    ax.set_xlabel("time step")
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## 3. Model-agnostic: PCI, one input channel at a time · 🟢 core

**Permutation Channel Importance** is the method of the PCI lab: destroy one input channel by **shuffling it across the test series**, leave the model alone, and measure how much the error grows.

There the output was one accuracy. Here the model forecasts 10 channels, so we get a **matrix**: row = the input channel we shuffled, column = the forecast channel whose error we measure.

**The known answer.** Each wave needs only itself (the diagonal). The follower needs the **leader** and not itself. The noise row must be zero.

> **The idea in words.** To test whether the model needs a channel, destroy that channel: give every series the channel of some other series. If the forecasts get worse, the model was using it.

In [ ]:
def permute_channel(Xin, c, rng):
    # A copy of Xin (N, T, C) in which channel c is shuffled across the N
    # series; all other channels stay in place.
    Xp = Xin.clone()
    # a random order of the N series
    order = torch.as_tensor(rng.permutation(len(Xin)), device=Xin.device)
    # ✏️ TODO 1 - give every series the channel c of ANOTHER series. Xin[i, :,
    # c] is channel c of series i.
    #   Replace BLANK with the random order of the series made one line above:
    #   order
    Xp[:, :, c] = Xin[BLANK, :, c]
    return Xp

### ✅ Check: `permute_channel`

In [ ]:
rng = np.random.default_rng(0)
Xp = permute_channel(X_test, 3, rng)
others = [c for c in range(D_IN) if c != 3]
assert Xp.shape == X_test.shape and torch.equal(
    Xp[:, :, others], X_test[:, :, others]
), "all channels except c must stay unchanged"
assert not torch.equal(
    Xp[:, :, 3], X_test[:, :, 3]
), "channel c is unchanged: did you assign the shuffled channel?"
assert torch.allclose(
    Xp[:, :, 3].sum(0), X_test[:, :, 3].sum(0), atol=1e-2
), "channel c must hold the same series, in a new order (shuffle whole series, not time steps)"
assert torch.equal(
    X_test, X[n_train + n_val :]
), "X_test itself must not change: work on the copy"
print("✅ permute_channel is right")

In [ ]:
def pci_matrix(repeats=3, seed=0):
    rng = np.random.default_rng(seed)
    M = np.zeros((D_IN, D_IN))
    for c in range(D_IN):
        for _ in range(repeats):
            # rise in MSE of every forecast channel
            M[c] += (
                channel_mse(permute_channel(X_test, c, rng)) - BASE
            ) / repeats
    return M


def show_matrix(ax, M, title, fmt="{:.2f}", cmap="Blues"):
    im = ax.imshow(M, cmap=cmap, vmin=0)
    ax.set_xticks(range(D_IN))
    ax.set_yticks(range(D_IN))
    ax.set_xticklabels(CHANNELS, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(CHANNELS, fontsize=8)
    for r in range(D_IN):
        for c in range(D_IN):
            if abs(M[r, c]) >= 0.05 * M.max():
                ax.text(
                    c,
                    r,
                    fmt.format(M[r, c]),
                    ha="center",
                    va="center",
                    fontsize=7,
                    color="white" if M[r, c] > 0.6 * M.max() else "k",
                )
    ax.set_xlabel("forecast channel")
    ax.set_ylabel("input channel")
    ax.set_title(title, fontsize=10)


PCI = pci_matrix()
fig, ax = plt.subplots(figsize=(5.6, 5))
show_matrix(ax, PCI, "PCI: rise in test MSE when the input channel is shuffled")
plt.tight_layout()
plt.show()
print(
    f"Follower forecast: shuffle the leader -> MSE +{PCI[LEADER, FOLLOWER]:.2f};  shuffle the follower itself -> MSE +{PCI[FOLLOWER, FOLLOWER]:.2f}"
)
print(
    f"Noise channel: largest rise in any forecast = {np.abs(PCI[NOISE]).max():.3f}"
)

## 4. Model-specific: Integrated Gradients for single forecasts · 🟢 core

PCI is **global**: one number per channel for the whole test set. IG is **local**: it explains one forecast of one series, and it can also say *which time step*.

`integrated_gradients` is the function you wrote in Sequence lab 1. The baseline is a history of zeros.

For one series and one forecast channel, IG gives a table of **100 time steps × 10 input channels**. To compare with PCI, add up the size of the attributions over time: one number per input channel. Do this for every forecast channel, average over a few series, and you get a matrix with the same layout as PCI.

To put the two side by side, each column is divided by its sum: *of everything this forecast's explanation points at, what share goes to each input channel?* The noise forecast is left blank: it is a constant, so there is nothing to explain.

> **The idea in words.** IG (from sequence lab 1) gives a value to every input number: 100 time steps × 10 channels. To compare with PCI we add the sizes of these values over time, which leaves one number per input channel.

In [ ]:
def integrated_gradients(f, x, baseline, steps=32):
    # f: maps a batch (B, T, C) to B numbers. x, baseline: one series each,
    # shape (T, C). Returns attributions, shape (T, C).
    alphas = (torch.arange(steps, device=x.device) + 0.5) / steps
    path = (baseline + alphas.view(-1, 1, 1) * (x - baseline)).requires_grad_(
        True
    )
    grads = torch.autograd.grad(f(path).sum(), path)[0]
    return ((x - baseline) * grads.mean(0)).detach()


def ig_channel_matrix(n_series=20):
    G = torch.zeros(D_IN, D_IN, device=DEVICE)
    for x in X_test[:n_series]:
        for d in range(D_IN):
            # ✏️ TODO 2a - the number IG explains: the forecast of ONE channel,
            # added over the 20 forecast steps.
            #   forecast(z) has shape (B, 20 steps, 10 channels). Replace BLANK
            #   with the forecast channel of this loop: d
            f = lambda z: forecast(z)[:, :, BLANK].sum(1)
            # (100 time steps, 10 input channels)
            a = integrated_gradients(f, x, torch.zeros_like(x))
            # ✏️ TODO 2b - one number per input channel: add |a| over the time
            # steps. a has shape (100 time steps,
            #   10 channels), so time is its first dimension. Replace BLANK with
            #   that dimension: 0
            G[:, d] += a.abs().sum(BLANK) / n_series
    return G.cpu().numpy()

### ✅ Check: the IG channel matrix

In [ ]:
IG = ig_channel_matrix()
assert (
    IG.shape == (D_IN, D_IN) and (IG >= 0).all()
), "the matrix must be 10 x 10 and non-negative (absolute attributions)"
x = X_test[0]
a = integrated_gradients(
    lambda z: forecast(z)[:, :, 4].sum(1), x, torch.zeros_like(x), steps=128
)
gap = float(
    forecast(x[None])[0, :, 4].sum()
    - forecast(torch.zeros_like(x)[None])[0, :, 4].sum()
)
assert abs(float(a.sum()) - gap) < 0.05 * max(
    1.0, abs(gap)
), "completeness fails: is f the forecast of channel d added over the steps?"
assert all(
    IG[:, d].argmax() == d for d in range(2, 9)
), "each wave's forecast should lean most on its own channel"
print("✅ the IG channel matrix is right")

In [ ]:
def share(M):
    # Each column divided by its sum: where does this forecast's explanation go?
    # Columns with almost nothing to explain are left blank.
    total = M.sum(0)
    return M / np.where(total > 0.02 * total.max(), total, np.inf)


fig, axes = plt.subplots(1, 2, figsize=(11.5, 5))
show_matrix(
    axes[0], share(np.clip(PCI, 0, None)), "PCI, as a share of each column"
)
show_matrix(
    axes[1], share(IG), "IG (mean |attribution|), as a share of each column"
)
plt.tight_layout()
plt.show()

# the channels each forecast can really use
TRUTH = np.eye(D_IN, dtype=bool)
TRUTH[LEADER, FOLLOWER] = True
TRUTH[FOLLOWER, FOLLOWER] = False
cols = [d for d in range(D_IN) if d != NOISE]
display(
    pd.DataFrame(
        {
            "PCI": [
                share(np.clip(PCI, 0, None))[:, d][TRUTH[:, d]].sum()
                for d in cols
            ],
            "IG": [share(IG)[:, d][TRUTH[:, d]].sum() for d in cols],
        },
        index=[CHANNELS[d] for d in cols],
    )
    .round(2)
    .T.rename_axis("share of the explanation on the right input channel")
)
print(
    f"Share on the noise input, averaged over the forecasts:  PCI {share(np.clip(PCI, 0, None))[NOISE, cols].mean():.3f}   IG {share(IG)[NOISE, cols].mean():.3f}"
)

## 5. Which moment? The 20-step delay as a known answer · 🟢 core

Now use what PCI cannot give: **time**. Explain the follower's forecast at **one** forecast step $h$ (0 to 19) and keep the attributions on the **leader** channel. The data says where they must be: the follower at step $100 + h$ is the leader at step $80 + h$.

So in a map with one row per forecast step and one column per input time step, a correct explanation is a **diagonal line** from (0, 80) to (19, 99).

For contrast, do the same for a plain wave (channel 4) and its own history: there is no delay to find, so the model should lean on the most recent steps.

> **What this cell does.** It asks IG *which moment* of the leader each forecast step of the follower uses.

In [ ]:
def ig_time_map(out_ch, in_ch, n_series=30):
    # Rows: forecast step h. Columns: input time step. Value: mean |IG| of input
    # channel in_ch for the forecast of out_ch at step h.
    Mh = torch.zeros(OUT_STEPS, IN_STEPS, device=DEVICE)
    for x in X_test[:n_series]:
        for h in range(OUT_STEPS):
            a = integrated_gradients(
                lambda z: forecast(z)[:, h, out_ch], x, torch.zeros_like(x)
            )
            Mh[h] += a[:, in_ch].abs() / n_series
    return Mh.cpu().numpy()


IG_FOLLOWER, IG_WAVE = ig_time_map(FOLLOWER, LEADER), ig_time_map(4, 4)


def show_time_map(ax, M, title, diagonal=True):
    ax.imshow(
        M / M.max(1, keepdims=True),
        cmap="Blues",
        aspect="auto",
        origin="lower",
        extent=[-0.5, IN_STEPS - 0.5, -0.5, OUT_STEPS - 0.5],
    )
    if diagonal:
        ax.plot(
            [IN_STEPS - LAG, IN_STEPS - 1],
            [0, OUT_STEPS - 1],
            color=RED,
            lw=1,
            ls="--",
            label="known answer: 80 + h",
        )
    ax.set_xlim(50, IN_STEPS - 0.5)
    ax.set_xlabel("input time step")
    ax.set_ylabel("forecast step h")
    ax.set_title(title, fontsize=10)


fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
show_time_map(
    axes[0], IG_FOLLOWER, "IG: follower forecast, attributions on the leader"
)
axes[0].legend(frameon=False, fontsize=8, loc="upper left")
show_time_map(
    axes[1],
    IG_WAVE,
    "IG: wave 4 forecast, attributions on wave 4",
    diagonal=False,
)
plt.tight_layout()
plt.show()
hit = (
    np.abs(IG_FOLLOWER.argmax(1) - (IN_STEPS - LAG + np.arange(OUT_STEPS))) <= 1
)
print(
    f"Follower: the IG peak is within 1 step of the known answer for {hit.sum()} of {OUT_STEPS} forecast steps"
)
print(
    "Wave 4:   the IG peak is at input step",
    sorted(set(IG_WAVE.argmax(1).tolist())),
)

> **What to notice.** Left: the dark cells follow the dashed line, so IG finds the 20-step delay. Right: a plain wave just looks at its own most recent steps.

## 6. Attention: does the map show the delay? · 🟢 core

The decoder's **cross-attention** is where forecast steps look at the history. Its weights have the right shape for an explanation: one row per forecast step, one column per input time step, each row adding up to 1.

Two limits are built in before we look:

- **One map for all channels.** A token is a whole time step (all 10 channels), so attention can say *when*, never *which channel*. The follower and wave 4 get the same map, although section 5 showed they use the history differently.
- **Four heads.** Each head has its own map. The usual picture averages them.

> **What this cell does.** It draws the attention weights of the decoder: for each forecast step (row), how much it looks at each input step (column). First the average of the 4 heads, then each head.

In [ ]:
with torch.no_grad():
    forecast(X_test[:500])
# (4 heads, 20 forecast steps, 100 input steps), averaged over 500 series
ATT = CROSS.last_attn.mean(0).cpu().numpy()

fig, axes = plt.subplots(1, 5, figsize=(18, 3.4), sharey=True)
show_time_map(axes[0], ATT.mean(0), "cross-attention, mean of the 4 heads")
for k in range(4):
    show_time_map(axes[k + 1], ATT[k], f"head {k}")
for ax in axes[1:]:
    ax.set_ylabel("")
axes[0].legend(frameon=False, fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

answer = IN_STEPS - LAG + np.arange(OUT_STEPS)
rows = {"mean of heads": ATT.mean(0), **{f"head {k}": ATT[k] for k in range(4)}}
display(
    pd.DataFrame(
        {
            name: {
                "forecast steps whose peak is within 1 step of 80 + h": int(
                    (np.abs(A.argmax(1) - answer) <= 1).sum()
                ),
                "forecast steps whose peak is in the last 5 inputs": int(
                    (A.argmax(1) >= IN_STEPS - 5).sum()
                ),
                "rank correlation with IG, follower": np.mean(
                    [
                        spearmanr(A[h], IG_FOLLOWER[h])[0]
                        for h in range(OUT_STEPS)
                    ]
                ),
                "rank correlation with IG, wave 4": np.mean(
                    [spearmanr(A[h], IG_WAVE[h])[0] for h in range(OUT_STEPS)]
                ),
            }
            for name, A in rows.items()
        }
    ).round(2)
)

> **What to notice.** The average map hides the delay. One single head follows the dashed line. Looking only at the average would mislead you.

## 7. Is the attention used? Replace the weights · 🟢 core

A map can look meaningful and still not matter (lab S10: on BiLSTM states, shuffling the attention changed few predictions). The test is the same here: **keep the model and the input, change only the cross-attention weights, and watch the error.**

- **Uniform:** every input step gets the same weight, 1/100.
- **Shuffled in time:** the same weights, moved to other input steps.
- **Shifted by 5:** every weight moves 5 steps into the past. If the model reads the leader at step $80 + h$ through attention, it now reads step $85 + h$: the wrong moment.

> **The idea in words.** To test whether the attention weights matter, replace them and watch the error. `uniform` makes all weights equal; `shuffled` moves them to random places; `shifted` moves them 5 steps into the past.

In [ ]:
def uniform(att):
    # att: (B, heads, queries, keys). The same weight for every key; each row
    # still adds up to 1.
    # ✏️ TODO 3a - every key gets the same weight: 1 / (number of keys). The
    # keys are the last dimension of att.
    #   Replace BLANK with the number of keys: att.shape[-1]
    return torch.full_like(att, 1.0 / BLANK)


def shuffled(att):
    # The same weights, moved to other keys (one random order for the whole
    # batch).
    # ✏️ TODO 3b - torch.randperm(n) gives a random order of n items; att[...,
    # order] moves the weights to other keys.
    #   Replace BLANK with the number of keys again: att.shape[-1]
    return att[..., torch.randperm(BLANK, device=att.device)]


def shifted(att, by=5):
    # Every weight moves `by` keys to the left, into the past (the first weights
    # wrap around to the end).
    return torch.roll(att, -by, dims=-1)

### ✅ Check: `uniform` and `shuffled`

In [ ]:
A0 = CROSS.last_attn[:4]
U, S = uniform(A0), shuffled(A0)
assert U.shape == A0.shape and torch.allclose(
    U, torch.full_like(A0, 1 / IN_STEPS)
), "uniform: every weight must be 1/100"
assert S.shape == A0.shape and torch.allclose(
    S.sum(-1), A0.sum(-1), atol=1e-5
), "shuffled: every row must still add up to 1"
assert torch.allclose(
    S.sort(-1).values, A0.sort(-1).values
) and not torch.equal(
    S, A0
), "shuffled: the same weights in each row, in a new order"
print("✅ uniform and shuffled are right")

In [ ]:
def mse_with(override):
    torch.manual_seed(0)
    CROSS.override = override
    try:
        return channel_mse(X_test)
    # always give the model its own attention back
    finally:
        CROSS.override = None


show = [LEADER, FOLLOWER, 4, NOISE]
table = pd.DataFrame(
    {
        "own attention": BASE,
        "uniform": mse_with(uniform),
        "shuffled in time": mse_with(shuffled),
        "shifted by 5": mse_with(shifted),
    },
    index=CHANNELS,
)
table.loc["mean of waves 2-8"] = table.iloc[2:9].mean()
display(
    table.loc[["0 leader", "1 follower", "mean of waves 2-8", "9 noise"]]
    .round(3)
    .rename_axis("test MSE")
)

## 8. What did you find?

When this notebook was built, the numbers were (they move with the random seed and between CPU and GPU; which head does what can change too):

| 2,000 test series | Result |
|---|---|
| Test MSE (predict the mean: about 1.2 for the waves) | follower 0.02, waves 0.21, leader 1.02, noise 1.00 |
| PCI, follower forecast | shuffle the leader: MSE +2.84; shuffle the follower itself: +0.00 |
| PCI, noise input | 0.000 for every forecast |
| IG, share of the explanation on the right input channel | follower 0.61, waves 0.57 to 0.68; noise input 0.005 |
| IG, peak within 1 step of 80 + h | 20 of 20 forecast steps |
| Attention, peak within 1 step of 80 + h | mean of heads: 3 of 20; one head (head 2): 20 of 20; another head (head 1) stays on the last 5 inputs |
| MSE with uniform attention | follower 1.23, waves 1.59 |
| MSE with attention shifted by 5 steps | follower 0.61, waves 0.43 |

Answer in one or two sentences each.

1. PCI and IG were both marked against the known answer. Where did each one put the follower's forecast? Which one gave the cleaner answer, and what can IG tell you that PCI cannot?
2. IG puts a small share of every forecast on channels that cannot matter. PCI does not. Give one reason. Think about what each method compares the input with.
3. Look at the heads one by one, then at their mean. What would you have concluded about the 20-step delay from the mean map alone?
4. In lab S10, shuffling the attention on BiLSTM states changed few predictions. Here, replacing the cross-attention breaks the forecast. What is different about where the attention sits in the two models?
5. So: is the cross-attention map an explanation of this model? Say what it gets right, and name two things it cannot tell you.

**Try it.**

- Set `LAG = 10` (and rerun everything). Do the IG diagonal and the attention head move with it?
- Train the model again with `torch.manual_seed(1)`. Is it the same head that tracks the delay? What does that say about reading meaning into "head 2"?
- Occlude a window of 5 time steps of the leader, as in Sequence lab 1, and plot the rise in the follower's error against the window position.

**Where this goes next.** The waves here were *generated* as trend + cycles + noise. Labs D20 (*STL*) and D22 (*N-BEATS, forecast by parts*) build that structure into the model, so the forecast can be read part by part with no explanation method at all.